In [47]:
import math
import pandas as pd
import numpy as np
import json

# READ DATA (and drop testing workerids)

In [111]:
df = pd.read_csv("./dataRaw/experiment_6323_results.csv")

unnecessary_cols = ["questionid", "filename", "listnumber", "assignmentid", "hitid", "origin", "timestamp", "partid", "id", "imgorder"]
unformatted_df = df.drop(unnecessary_cols, axis=1)

testworkers = ["test", "123", "testFirefox", "testRedirect", "testFinal"]  # add more to this list if there were more test runs
unformatted_df = unformatted_df[~unformatted_df["workerid"].isin(testworkers)]

len(unformatted_df["workerid"].unique())

78

## FORMAT RATING VALUES FROM THE ANSWER COLUMN

In [61]:
def format_answer(answer, weak_word, strong_word, antonym):
    """
    returns multiple results.
    whatever data is needed out of the answer, edit and manipulate it in this function
    """
    results = json.loads(json.loads(answer))
    new_dict = dict()
    
    if not results.get("ratings"):
        return None, None, None, None, None
    for item in results["ratings"]:
        name = item["image"].replace(".png", "")
        new_dict.update(
            {name: item["rating"]}
        )

    if results["speaker"] in ["frank", "bri"]:
        speakerType = "native"
    else:
        speakerType = "nonnative"
    
    if results["speaker"] in ["bri", "sasha"]:
        speakerGender = "female"
    else:
        speakerGender = "male"

    # weak word rating, strong word rating, antonym rating
    return new_dict.get(weak_word), new_dict.get(strong_word), new_dict.get(antonym), speakerType, speakerGender

In [62]:
answers_only_df = unformatted_df.apply(
    lambda x: format_answer(x.answer, x.weak, x.strong, x.antonym), 
    axis=1, 
    result_type="expand",  # this is how you make it into multiple columns
)
answers_only_df.columns=["derivationStrength", "strongRating", "antonymRating", "speakerType", "speakerGender"]
answers_only_df

,derivationStrength,strongRating,antonymRating,speakerType,speakerGender
1,90.0,10.0,0.0,nonnative,male
2,35.0,60.0,5.0,native,female
3,94.0,6.0,0.0,nonnative,male
4,100.0,0.0,0.0,nonnative,male
5,100.0,0.0,0.0,native,female
...,...,...,...,...,...
2712,100.0,0.0,0.0,nonnative,female
2713,100.0,0.0,0.0,native,female
2714,100.0,0.0,0.0,nonnative,female
2715,100.0,0.0,0.0,native,male


## READ WORD FREQ DICT IN

In [63]:
word_freq_dict_df = pd.read_csv("./dataRaw/wordFreqDict.csv")
word_freq_dict_df.head()

word_freq_dict_df[word_freq_dict_df['Frequency'].isin(["x"])]

,Word,Rank,Frequency


## RENAME COLUMN NAMES TO WHAT THE MODEL NEEDS

In [113]:
answers_df = pd.concat([unformatted_df, answers_only_df], axis=1)
answers_df = answers_df.drop(["answer"], axis=1)

answers_df = answers_df.rename(columns={
    "workerid": "participantId",
    "itemid": "itemId",
    "type": "itemType"
})

answers_df[answers_df["participantId"] == "66b153e252bf568e5c9f0ba2"]

,participantId,itemId,itemType,weak,strong,antonym,lowfreq1,lowfreq2,derivationStrength,strongRating,antonymRating,speakerType,speakerGender


# FILTER - EXCLUSION CRITERIA

### Picture Rating Control Trials

In [119]:
# filter out rating scores
#    - if avg. rating across all unambiguous trials were < 80%
umambiguous_trials = answers_df[answers_df["itemType"] == "unambiguous"]
picrating_filtered_participants = list(
    pd.unique(
        umambiguous_trials.groupby("participantId").filter(
            lambda x: x["derivationStrength"].mean() < 80
        )["participantId"]
    )
)
picrating_filtered_participants

[]

### Picture Rating Technical Problems

In [105]:
# a lot of responses for one person didn't get recorded unfortunately
countNaN = umambiguous_trials["derivationStrength"].isnull().groupby(
    umambiguous_trials["participantId"]
).sum().astype(int).reset_index(name="countNaN")

technical_prob_filtered_participants = list(countNaN[countNaN["countNaN"] > 1]["participantId"].unique())
technical_prob_filtered_participants

['69cc1fac2b52b1c4dd21cc7d']

## ToM Filtered Participants

Get a list from the R script:
- technical problems (anything over 10% of responses missing)
- control accuracy < 80%

In [120]:
nathan4u_filtered_participants = [
    # technical issues: 50+ unanswered
    "69c093d33989aa791c4e8b2d", 
    "6a2482e9af72b55296872b0d",
    "6a00dbcc626a780a8e5c07a4",
    "6a232745f21e165e2b54ed58",

    # control accuracy < 80%
    "6965bd850ce22095dd7a84ca",
    "6978d830c5d415696508430d",
    "69a6eb2bc679065ed1cecc51",
    "69ba37faee13f176e7635323",
    "69c093d33989aa791c4e8b2d",
    "69cc1fac2b52b1c4dd21cc7d",
    "69f674db81cb46678aae6391",
    "69fb86296fb6f0919bf31075",
    "6a00dbcc626a780a8e5c07a4",
    "6a14434573bcee3f17b7f762",
    "6a14e67cf20909f60e34ae64",
    "6a232745f21e165e2b54ed58",
    "6a2482e9af72b55296872b0d",
]

## Audio Filtered Participants

After transcribing the audio for the two practice perspective taking trials, any that did not choose the target word will be excluded.

Anyone who did not provide a response that is indicative of understanding the task in the final critical audio explanation will be excluded.

In [121]:
audio_filtered_participants = []

## Total Filtered Participants

In [122]:
all_filtered_participants = list(set(
    picrating_filtered_participants + 
    nathan4u_filtered_participants + 
    audio_filtered_participants + 
    technical_prob_filtered_participants
))

print("total number of filtered participants: ", len(all_filtered_participants))
all_filtered_participants

total number of filtered participants:  13


['6978d830c5d415696508430d',
 '6a14434573bcee3f17b7f762',
 '69cc1fac2b52b1c4dd21cc7d',
 '6965bd850ce22095dd7a84ca',
 '6a232745f21e165e2b54ed58',
 '69a6eb2bc679065ed1cecc51',
 '6a2482e9af72b55296872b0d',
 '6a14e67cf20909f60e34ae64',
 '69c093d33989aa791c4e8b2d',
 '69ba37faee13f176e7635323',
 '69f674db81cb46678aae6391',
 '69fb86296fb6f0919bf31075',
 '6a00dbcc626a780a8e5c07a4']

# MERGE DFs TOGETHER

## all the word information back into the response dataframe

## read and merge the ToM scores in

In [115]:
# TODO: do this

In [110]:
final_remove_cols = ["weakRank", "strongRank", "antonymRank", "weakFreq", "strongFreq", "antonymFreq"]
final_df = final_df.drop(final_remove_cols, axis=1)
final_df.to_csv("./dataOutput/formatted_results.csv")

# TRIAL STUFF

In [18]:
unformatted_df["answer"].head()

0    "{\"prob1\":\"17\",\"prob2\":0,\"prob3\":83,\"...
1    "{\"prob1\":0,\"prob2\":0,\"prob3\":\"100\",\"...
2    "{\"prob1\":5,\"prob2\":\"95\",\"prob3\":0,\"p...
3    "{\"prob1\":0,\"prob2\":\"90\",\"prob3\":10,\"...
4    "{\"prob1\":0,\"prob2\":97,\"prob3\":\"3\",\"p...
Name: answer, dtype: object

In [7]:
sample = unformatted_df.iloc[0]
sample

workerid                                               19970528
questionid                                               116506
answer        "{\"prob1\":\"17\",\"prob2\":0,\"prob3\":83,\"...
itemid                                                        1
type                                                   critical
weak                                                       soft
strong                                                    mushy
antonym                                                 crunchy
lowfreq1                                               panicked
lowfreq2                                          introspective
imgorder                                                 random
Name: 0, dtype: object

In [8]:
sample_answer = json.loads(json.loads(sample["answer"]))
sample_answer

{'prob1': '17',
 'prob2': 0,
 'prob3': 83,
 'probsum': 100,
 'ratings': [{'image': 'mushy.png', 'rating': 17},
  {'image': 'crunchy.png', 'rating': 0},
  {'image': 'soft.png', 'rating': 83}],
 'speaker': 'frank',
 'timeTaken': 32160.69999998808}

In [14]:
new_dict = dict()
for item in sample_answer["ratings"]:
    name = item["image"].replace(".png", "")

    # if name == sample["weak"]:
    #     tag = "target"
    # elif name == sample["strong"]:
    #     tag = "strong"
    # else:
    #     tag = "antonym"
    
    new_dict.update(
        {name: item["rating"]}
    )
new_dict

{'mushy': 17, 'crunchy': 0, 'soft': 83}

In [13]:
if sample_answer["speaker"] in ["frank", "bri"]:
    speakerType = "native"
else:
    speakerType = "nonnative"

if sample_answer["speaker"] in ["bri", "sasha"]:
    speakerGender = "female"
else:
    speakerGender = "male"

speakerGender, speakerType, sample["type"]

('male', 'native', 'critical')

In [12]:
word_freq_dict = pd.read_csv("wordFreqDict.csv")
word_freq_dict.head()

,Word,Rank,Frequency
0,accurate,2754,11842
1,additive,15315,725
2,alive,1541,24184
3,ancient,1832,19818
4,anecdotal,10575,1424
